# CSE-CIC-IDS2018 → Cleaned 40k Samples Each

Converts the raw **CSE-CIC-IDS2018** dataset into a cleaned, balanced version:
- ✅ Excludes date **20/2**
- ✅ Up to **40,000 samples per label** per file
- ✅ Removes non-numeric and `inf` values
- ✅ Globally shuffled
- ✅ Label column integer-encoded

> **Dataset source:** [Kaggle — CSE-CIC-IDS2018](https://www.kaggle.com/datasets/solarmainframe/ids-intrusion-csv)
>
> The cells below will mount Google Drive, install the Kaggle API, download the dataset, and run the full preprocessing pipeline.

## 1. Mount Google Drive
We'll use Drive to persist the output CSV so it survives session resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = "/content/drive/MyDrive/IDS2018"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output will be saved to: {OUTPUT_DIR}")

## 2. Set Up Kaggle API & Download Dataset

Reads `kaggle.json` directly from `MyDrive/IDS2018/` — no upload needed.

In [ ]:
import shutil

# Copy kaggle.json from Drive to where the Kaggle CLI expects it
os.makedirs("/root/.config/kaggle", exist_ok=True)
shutil.copy("/content/drive/MyDrive/IDS2018/kaggle.json", "/root/.config/kaggle/kaggle.json")
os.chmod("/root/.config/kaggle/kaggle.json", 0o600)
print("kaggle.json installed from Drive.")

In [ ]:
!pip install -q kaggle

# Download & unzip into /content/ids2018
!kaggle datasets download -d solarmainframe/ids-intrusion-csv -p /content/ids2018 --unzip
print("Download complete.")

## 3. Configuration

In [ ]:
INPUT_DIR         = "/content/ids2018"
OUTPUT_FILE       = os.path.join(OUTPUT_DIR, "CSE-CIC-IDS2018_Cleaned_40k.csv")
SAMPLES_PER_LABEL = 40_000
LABEL_COL         = "Label"
EXCLUDE_DATE      = "20-02"   # skip files whose name contains this string
RANDOM_SEED       = 42

## 4. Imports

In [ ]:
import glob
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

print(f"pandas : {pd.__version__}")
print(f"numpy  : {np.__version__}")

## 5. Helper Functions

In [ ]:
def load_and_tag(path: str) -> pd.DataFrame:
    """Load one CSV, strip column names, drop non-feature columns."""
    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.str.strip()
    for col in ["Timestamp", "timestamp", "Dst IP", "Src IP", "Flow ID"]:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)
    return df


def clean(df: pd.DataFrame) -> pd.DataFrame:
    """
    - Drop rows with missing Label
    - Coerce features to numeric (non-parseable → NaN)
    - Replace ±inf with NaN
    - Drop remaining NaN rows
    """
    df = df.dropna(subset=[LABEL_COL])
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

    feat_cols = [c for c in df.columns if c != LABEL_COL]
    df[feat_cols] = df[feat_cols].apply(pd.to_numeric, errors="coerce")
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    return df


def sample_per_label(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    """Take up to n rows per unique label value."""
    parts = [
        grp.sample(n=min(len(grp), n), random_state=seed)
        for _, grp in df.groupby(LABEL_COL, sort=False)
    ]
    return pd.concat(parts, ignore_index=True)


print("Helper functions defined.")

## 6. Discover CSV Files

In [ ]:
csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "**", "*.csv"), recursive=True))
csv_files = [f for f in csv_files if EXCLUDE_DATE not in os.path.basename(f)]

print(f"Found {len(csv_files)} CSV file(s) after excluding '{EXCLUDE_DATE}':")
for f in csv_files:
    print(f"  {os.path.basename(f)}")

## 7. Process Files One-by-One (Memory-Safe)

In [ ]:
all_samples = []

for fp in csv_files:
    fname = os.path.basename(fp)
    print(f"\n→ Processing: {fname}")

    try:
        df = load_and_tag(fp)
    except Exception as e:
        print(f"  [SKIP] Could not read file: {e}")
        continue

    if LABEL_COL not in df.columns:
        print(f"  [SKIP] '{LABEL_COL}' column not found.")
        continue

    df = clean(df)
    print(f"  After cleaning : {len(df):,} rows")
    print(f"  Labels found   : {df[LABEL_COL].unique().tolist()}")

    sampled = sample_per_label(df, SAMPLES_PER_LABEL, RANDOM_SEED)
    print(f"  After sampling : {len(sampled):,} rows")

    all_samples.append(sampled)
    del df, sampled   # free RAM before next file

print("\n✓ All files processed.")

## 8. Combine & Global Shuffle

In [ ]:
print("Combining all sampled files …")
combined = pd.concat(all_samples, ignore_index=True)
del all_samples

combined = combined.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"Combined shape  : {combined.shape}")
print(f"\nLabel distribution (raw):\n{combined[LABEL_COL].value_counts()}")

## 9. Encode Labels

In [ ]:
le = LabelEncoder()
combined[LABEL_COL] = le.fit_transform(combined[LABEL_COL])

print("Label encoding map:")
for idx, cls in enumerate(le.classes_):
    count = (combined[LABEL_COL] == idx).sum()
    print(f"  {idx:2d} → {cls:<40s} ({count:,} samples)")

## 10. Preview

In [ ]:
print(f"Final dataset shape: {combined.shape}")
combined.head()

## 11. Save to Google Drive

In [ ]:
combined.to_csv(OUTPUT_FILE, index=False)
print(f"✓ Saved → {OUTPUT_FILE}")
print(f"  Rows    : {combined.shape[0]:,}")
print(f"  Columns : {combined.shape[1]}")